# COMP9501 团队运行入口：中文版

请使用本人已授权的 GitHub/Google 账号打开 private notebook；也可以从仓库下载该 ipynb 后上传到 Colab。源码快照已内嵌，runtime 中不需要 GitHub token、git clone 或 MCP。GPU 分配和订阅权益属于各自的运行时账号。完整说明见仓库 README.zh-CN.md 与 COLLABORATION.md。

首次选择“全部运行 / Run all”会校验并展开源码、安装固定依赖、执行离线检查。恢复权重、训练、评价和结果导出由下面的开关显式启用。修改源码请通过仓库分支/PR，再重新生成 launcher；这里的 runtime 目录是执行副本。


In [ ]:
# Project source, checked before importing. Do not edit the encoded payload.
from pathlib import Path
import base64, hashlib, os, sys, subprocess, uuid
SOURCE_SNAPSHOT = '89a0fd982dff73cd32c2bfe3031dd6b50edf3de7d6e6e85d6a40d466e3d874c3'
_helper_bytes = base64.b64decode('IiIiQ3JlZGVudGlhbC1mcmVlIHNvdXJjZSBpbXBvcnQsIGNoZWNrZWQgY2hlY2twb2ludCByZXN0b3JlIGFuZCBydW4gZXhwb3J0LiIiIgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgsUHVyZVBvc2l4UGF0aAppbXBvcnQgaGFzaGxpYixpbyxqc29uLG9zLHN0YXQsemlwZmlsZQoKCmRlZiBzaGEyNTZfZmlsZShwYXRoKToKICAgIGg9aGFzaGxpYi5zaGEyNTYoKQogICAgd2l0aCBQYXRoKHBhdGgpLm9wZW4oJ3JiJykgYXMgc3RyZWFtOgogICAgICAgIGZvciBibG9jayBpbiBpdGVyKGxhbWJkYTpzdHJlYW0ucmVhZCgxMDI0KjEwMjQpLGInJyk6aC51cGRhdGUoYmxvY2spCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQoKCmRlZiBfZW50cmllcyhhcmNoaXZlLG1heF9ieXRlcyk6CiAgICBlbnRyaWVzPWFyY2hpdmUuaW5mb2xpc3QoKTtuYW1lcz1baS5maWxlbmFtZSBmb3IgaSBpbiBlbnRyaWVzXQogICAgaWYgbGVuKGVudHJpZXMpPjUwMDAgb3IgbGVuKG5hbWVzKSE9bGVuKHNldChuYW1lcykpOnJhaXNlIFZhbHVlRXJyb3IoJ0R1cGxpY2F0ZSBvciBleGNlc3NpdmUgWklQIGVudHJpZXMnKQogICAgaWYgc3VtKGkuZmlsZV9zaXplIGZvciBpIGluIGVudHJpZXMpPm1heF9ieXRlczpyYWlzZSBWYWx1ZUVycm9yKCdaSVAgZXhjZWVkcyB1bmNvbXByZXNzZWQtc2l6ZSBsaW1pdCcpCiAgICBmb3IgaXRlbSBpbiBlbnRyaWVzOgogICAgICAgIG5hbWU9aXRlbS5maWxlbmFtZTtwYXRoPVB1cmVQb3NpeFBhdGgobmFtZSkKICAgICAgICBpZiBjaHIoOTIpIGluIG5hbWU6cmFpc2UgVmFsdWVFcnJvcignQmFja3NsYXNoIGluIFpJUCBtZW1iZXInKQogICAgICAgIGlmIGNocig5MikgaW4gbmFtZTpyYWlzZSBWYWx1ZUVycm9yKCdCYWNrc2xhc2ggaW4gWklQIG1lbWJlcicpCiAgICAgICAgaWYgaXRlbS5pc19kaXIoKSBvciBwYXRoLmlzX2Fic29sdXRlKCkgb3IgJy4uJyBpbiBwYXRoLnBhcnRzIG9yICdcXCcgaW4gbmFtZSBvciAnOicgaW4gbmFtZSBvciBub3QgcGF0aC5wYXJ0cyBvciBwYXRoLmFzX3Bvc2l4KCkhPW5hbWU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0ludmFsaWQgYXJjaGl2ZSBtZW1iZXInKQogICAgICAgIGlmIHN0YXQuU19JU0xOSyhpdGVtLmV4dGVybmFsX2F0dHI+PjE2KTpyYWlzZSBWYWx1ZUVycm9yKCdTeW1saW5rIGFyY2hpdmUgbWVtYmVyJykKICAgIHJldHVybiBlbnRyaWVzCgoKZGVmIF9yZXN0b3JlKGFyY2hpdmUsZmlsZXMsZGVzdGluYXRpb24sbWF4X2J5dGVzKToKICAgIGRlc3RpbmF0aW9uPVBhdGgoZGVzdGluYXRpb24pLnJlc29sdmUoKTtlbnRyaWVzPV9lbnRyaWVzKGFyY2hpdmUsbWF4X2J5dGVzKQogICAgaWYge2UuZmlsZW5hbWUgZm9yIGUgaW4gZW50cmllc30hPXNldChmaWxlcyk6cmFpc2UgVmFsdWVFcnJvcignQXJjaGl2ZSBmaWxlIGxpc3QgZG9lcyBub3QgbWF0Y2ggbWFuaWZlc3QnKQogICAgIyBWYWxpZGF0ZSBldmVyeXRoaW5nIGJlZm9yZSBjcmVhdGluZyBhIGZpbGUsIGluY2x1ZGluZyBjb2xsaXNpb24gY2hlY2tzLgogICAgZm9yIGl0ZW0gaW4gZW50cmllczoKICAgICAgICBjb250ZW50PWFyY2hpdmUucmVhZChpdGVtKTtleHBlY3RlZD1maWxlc1tpdGVtLmZpbGVuYW1lXQogICAgICAgIGlmIGhhc2hsaWIuc2hhMjU2KGNvbnRlbnQpLmhleGRpZ2VzdCgpIT1leHBlY3RlZDpyYWlzZSBWYWx1ZUVycm9yKCdBcmNoaXZlIG1lbWJlciBjaGVja3N1bSBtaXNtYXRjaCcpCiAgICAgICAgdGFyZ2V0PWRlc3RpbmF0aW9uL2l0ZW0uZmlsZW5hbWUKICAgICAgICBpZiBub3QgdGFyZ2V0LnJlc29sdmUoKS5pc19yZWxhdGl2ZV90byhkZXN0aW5hdGlvbik6cmFpc2UgVmFsdWVFcnJvcignUGF0aCBlc2NhcGVzIGRlc3RpbmF0aW9uJykKICAgICAgICBpZiB0YXJnZXQuZXhpc3RzKCkgYW5kIChub3QgdGFyZ2V0LmlzX2ZpbGUoKSBvciBzaGEyNTZfZmlsZSh0YXJnZXQpIT1leHBlY3RlZCk6CiAgICAgICAgICAgIHJhaXNlIEZpbGVFeGlzdHNFcnJvcignRXhpc3RpbmcgY29udGVudCBkaWZmZXJzOiAnK2l0ZW0uZmlsZW5hbWUpCiAgICBmb3IgaXRlbSBpbiBlbnRyaWVzOgogICAgICAgIHRhcmdldD1kZXN0aW5hdGlvbi9pdGVtLmZpbGVuYW1lCiAgICAgICAgaWYgbm90IHRhcmdldC5leGlzdHMoKToKICAgICAgICAgICAgdGFyZ2V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSk7dGFyZ2V0LndyaXRlX2J5dGVzKGFyY2hpdmUucmVhZChpdGVtKSkKICAgIHJldHVybiBkZXN0aW5hdGlvbgoKCmRlZiBtYXRlcmlhbGl6ZV9zb3VyY2UocGF5bG9hZCxleHBlY3RlZF9zaGEsZGVzdGluYXRpb24pOgogICAgaWYgaGFzaGxpYi5zaGEyNTYocGF5bG9hZCkuaGV4ZGlnZXN0KCkhPWV4cGVjdGVkX3NoYTpyYWlzZSBWYWx1ZUVycm9yKCdTb3VyY2UgYnVuZGxlIGNoZWNrc3VtIG1pc21hdGNoJykKICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKGlvLkJ5dGVzSU8ocGF5bG9hZCkpIGFzIGFyY2hpdmU6CiAgICAgICAgX2VudHJpZXMoYXJjaGl2ZSw2NCoxMDI0KjEwMjQpCiAgICAgICAgbWFuaWZlc3Q9anNvbi5sb2FkcyhhcmNoaXZlLnJlYWQoJ2J1bmRsZS1tYW5pZmVzdC5qc29uJykpCiAgICAgICAgZmlsZXM9bWFuaWZlc3RbJ2ZpbGVzJ107ZmlsZXNbJ2J1bmRsZS1tYW5pZmVzdC5qc29uJ109aGFzaGxpYi5zaGEyNTYoYXJjaGl2ZS5yZWFkKCdidW5kbGUtbWFuaWZlc3QuanNvbicpKS5oZXhkaWdlc3QoKQogICAgICAgIHJldHVybiBfcmVzdG9yZShhcmNoaXZlLGZpbGVzLGRlc3RpbmF0aW9uLDY0KjEwMjQqMTAyNCkKCgpkZWYgcmVzdG9yZV9jaGVja3BvaW50cyhhcmNoaXZlX3BhdGgsbWFuaWZlc3RfcGF0aCxkZXN0aW5hdGlvbik6CiAgICBtYW5pZmVzdD1qc29uLmxvYWRzKFBhdGgobWFuaWZlc3RfcGF0aCkucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpKQogICAgaWYgc2hhMjU2X2ZpbGUoYXJjaGl2ZV9wYXRoKSE9bWFuaWZlc3RbJ3NoYTI1NiddOnJhaXNlIFZhbHVlRXJyb3IoJ0NoZWNrcG9pbnQgWklQIGNoZWNrc3VtIG1pc21hdGNoJykKICAgIGlmIG5vdCBhbGwobmFtZS5zdGFydHN3aXRoKCdvdXRwdXRzLycpIGZvciBuYW1lIGluIG1hbmlmZXN0WydmaWxlcyddKTpyYWlzZSBWYWx1ZUVycm9yKCdDaGVja3BvaW50IG1hbmlmZXN0IG11c3QgY29udGFpbiBvdXRwdXRzIG9ubHknKQogICAgd2l0aCB6aXBmaWxlLlppcEZpbGUoYXJjaGl2ZV9wYXRoKSBhcyBhcmNoaXZlOgogICAgICAgIHJldHVybiBfcmVzdG9yZShhcmNoaXZlLG1hbmlmZXN0WydmaWxlcyddLGRlc3RpbmF0aW9uLDUxMioxMDI0KjEwMjQpCgoKZGVmIGV4cG9ydF9ydW4ocHJvamVjdF9yb290LHJ1bl9kaXJlY3RvcnksZGVzdGluYXRpb24sbWV0YWRhdGEpOgogICAgcm9vdD1QYXRoKHByb2plY3Rfcm9vdCkucmVzb2x2ZSgpO3J1bj1QYXRoKHJ1bl9kaXJlY3RvcnkpLnJlc29sdmUoKTtkZXN0PVBhdGgoZGVzdGluYXRpb24pLnJlc29sdmUoKQogICAgaWYgbm90IHJ1bi5pc19yZWxhdGl2ZV90byhyb290LydvdXRwdXRzJykgb3Igbm90IHJ1bi5pc19kaXIoKTpyYWlzZSBWYWx1ZUVycm9yKCdDaG9vc2UgYW4gZXhpc3Rpbmcgb3V0cHV0cyBydW4gZGlyZWN0b3J5JykKICAgIGlmIGRlc3QuaXNfcmVsYXRpdmVfdG8ocnVuKTpyYWlzZSBWYWx1ZUVycm9yKCdFeHBvcnQgWklQIG11c3QgYmUgb3V0c2lkZSB0aGUgc2VsZWN0ZWQgcnVuJykKICAgIGZpbGVzPXNvcnRlZChwIGZvciBwIGluIHJ1bi5yZ2xvYignKicpIGlmIHAuaXNfZmlsZSgpKQogICAgaWYgbm90IGZpbGVzOnJhaXNlIFZhbHVlRXJyb3IoJ1J1biBkaXJlY3RvcnkgaXMgZW1wdHknKQogICAgaWYgYW55KG5vdCBwLnJlc29sdmUoKS5pc19yZWxhdGl2ZV90byhydW4pIGZvciBwIGluIGZpbGVzKTpyYWlzZSBWYWx1ZUVycm9yKCdTeW1saW5rIGxlYXZlcyBydW4gZGlyZWN0b3J5JykKICAgIHJlY29yZD17KiptZXRhZGF0YSwnZmlsZXMnOntwLnJlbGF0aXZlX3RvKHJvb3QpLmFzX3Bvc2l4KCk6c2hhMjU2X2ZpbGUocCkgZm9yIHAgaW4gZmlsZXN9fQogICAgZGVzdC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShkZXN0LCd4Jyxjb21wcmVzc2lvbj16aXBmaWxlLlpJUF9ERUZMQVRFRCkgYXMgYXJjaGl2ZToKICAgICAgICBmb3IgcCBpbiBmaWxlczoKICAgICAgICAgICAgYXJjaGl2ZS53cml0ZShwLHAucmVsYXRpdmVfdG8ocm9vdCkuYXNfcG9zaXgoKSkKICAgICAgICBhcmNoaXZlLndyaXRlc3RyKCdydW4tbWV0YWRhdGEuanNvbicsanNvbi5kdW1wcyhyZWNvcmQsaW5kZW50PTIpKydcbicpCiAgICByZXR1cm4geydwYXRoJzpzdHIoZGVzdCksJ3NoYTI1Nic6c2hhMjU2X2ZpbGUoZGVzdCksJ2J5dGVzJzpkZXN0LnN0YXQoKS5zdF9zaXplfQo=')
assert hashlib.sha256(_helper_bytes).hexdigest() == '63285657e6c57bdb5da9acb23bb0831068a58594c645b6b02150bf3059be8d1f'
_bootstrap = {}
exec(compile(_helper_bytes, '<verified project bootstrap>', 'exec'), _bootstrap)
ROOT = Path('/content') / ('COMP9501-GP-' + SOURCE_SNAPSHOT[:12])
_bootstrap['materialize_source'](base64.b64decode('...'), SOURCE_SNAPSHOT, ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'scripts'))
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
print('Project source verified:', SOURCE_SNAPSHOT)
print('Runtime folder:', ROOT)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


In [ ]:
checks = [('code', 'test_math.py'), ('code', 'test_protocol.py'), ('code', 'test_independent_protocol.py'), ('code', 'test_activation_extension.py'), ('code', 'test_robustness_controls.py'), ('code/functional_merge', 'test_maths.py'), ('code/coefficient_merge', 'test_core.py'), ('code/merge_study', 'test_engine.py'), ('part1-reproduction-20261025/code', 'test_csa*.py'), ('scripts', 'test_colab_bundle.py')]
for folder, pattern in checks:
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', folder, '-p', pattern], check=True)


## 可选：恢复已有的 12 个 adapter

从组内授权文件存储获取 checkpoint ZIP；可以通过 Colab Files 上传，或填写本人已授权 Drive mount 下的路径。GitHub collaborator 权限不自动授予 Drive 文件访问权限。

填写 CHECKPOINT_ARCHIVE 后运行下一格；留空则跳过。恢复器会核对整个 ZIP 和逐文件 SHA，并拒绝覆盖不同内容。权重包不放进 Git，base model 与公开数据仍按原发布方下载。


In [ ]:
CHECKPOINT_ARCHIVE = ''  # e.g. path to the uploaded ZIP; leave blank to skip
if CHECKPOINT_ARCHIVE:
    from colab_bundle import restore_checkpoints
    restore_checkpoints(CHECKPOINT_ARCHIVE, ROOT / 'assets/checkpoints-2026-10-25.json', ROOT)
    print('Checkpoint archive and every restored file verified.')


## 可选：训练一组新的 adapter bank

先切换到 GPU，再启用训练。MEMBER 填自己的简短标识，SEED 选本次训练 seed，RUN_TRAINING=True 才实际启动。脚本会使用带随机标识的新输出目录，避免覆盖旧实验。

MEMBER 只是运行记录标签，可以使用昵称或实验编号，无需填写真实姓名。原任务的 calibration/validation 探索边界保持不变。只想复评时，保持训练开关关闭。


In [ ]:
MEMBER = 'member'  # use your own short alias
SEED = 42
RUN_TRAINING = False
RUN_NAME = MEMBER + '-bank-s' + str(SEED) + '-' + uuid.uuid4().hex[:8]
RUN_DIR = ROOT / 'outputs' / RUN_NAME
if RUN_TRAINING:
    assert not RUN_DIR.exists()
    subprocess.run([sys.executable, 'code/run_bank.py', '--config', 'configs/protocol-v2-a100.json', '--seed', str(SEED), '--output', str(RUN_DIR)], check=True)


## 可选：复评一个已恢复的 adapter

先恢复 checkpoint，设置 SEED 与 EVAL_TASK（sst2 / mrpc / rte / cola），将 RUN_EVALUATION=True。评价保留原任务头和原 source validation 协议，结果写入新的 run 目录。

仍需下载公开 base model 和数据；这一步不会重新训练 adapter。若只做评价，保持 RUN_TRAINING=False。


In [ ]:
RUN_EVALUATION = False
EVAL_TASK = 'sst2'  # sst2, mrpc, rte or cola
if RUN_EVALUATION:
    bank = ROOT / ('outputs/bank-v2-a100-s42' if SEED == 42 else 'outputs/replication-s43-s44/bank-s' + str(SEED))
    adapter = bank / EVAL_TASK / 'adapter'
    assert (adapter / 'adapter_model.safetensors').is_file(), 'Restore the checkpoint asset first.'
    RUN_NAME = MEMBER + '-eval-' + EVAL_TASK + '-s' + str(SEED) + '-' + uuid.uuid4().hex[:8]
    RUN_DIR = ROOT / 'outputs' / RUN_NAME
    RUN_DIR.mkdir(parents=True)
    subprocess.run([sys.executable, 'code/evaluate.py', '--config', 'configs/protocol-v2-a100.json', '--task', EVAL_TASK, '--adapter', str(adapter), '--output', str(RUN_DIR / 'result.json')], check=True)


## 可选：恢复或重新训练独立协议 v3

Part 2 最终协议使用单独的 v3 checkpoint ZIP，不是上方旧 bank 的 ZIP。INDEPENDENT_CHECKPOINT_ARCHIVE 填上传路径可校验恢复；或将 RUN_FRESH_V3=True 从头训练十二个 teacher 并评价固定权重基线，两条路线只能选一条。恢复会校验文件清单、迁移分区路径，不重新训练 teacher。默认两者都不执行；重新训练需要更多 GPU 时间。


In [ ]:
INDEPENDENT_CHECKPOINT_ARCHIVE = ''  # uploaded v3 checkpoint ZIP, or leave blank
RUN_FRESH_V3 = False
assert not (INDEPENDENT_CHECKPOINT_ARCHIVE and RUN_FRESH_V3), 'Choose restore or fresh training.'
V3_RUN_NAME = MEMBER + '-v3-' + uuid.uuid4().hex[:8]
V3_STUDY = ROOT / 'outputs' / V3_RUN_NAME
if INDEPENDENT_CHECKPOINT_ARCHIVE:
    subprocess.run([sys.executable, 'code/restore_independent_backup.py', '--archive', INDEPENDENT_CHECKPOINT_ARCHIVE, '--output', str(V3_STUDY)], check=True)
elif RUN_FRESH_V3:
    subprocess.run([sys.executable, 'code/run_independent_study.py', '--output', str(V3_STUDY)], check=True)
    subprocess.run([sys.executable, 'code/independent_baselines.py', '--study', str(V3_STUDY)], check=True)
if INDEPENDENT_CHECKPOINT_ARCHIVE or RUN_FRESH_V3:
    RUN_NAME, RUN_DIR = V3_RUN_NAME, V3_STUDY


## 可选：复现直接 activation 与路由对照

先恢复或完成上面的 v3 study，再启用 RUN_ACTIVATION_CONTROLS。程序执行五个静态目标及两个学习 router，保留锁定的 fit、selection 与 final 权限。需要至少 20 GiB 显存的 CUDA GPU，本次恢复重跑使用 L4。输出目录每次新建，离开临时 VM 前立即导出。

该最终分区此前已用于探索；重复同一组三个 seed 不增加独立样本数，也不构成全新盲测。


In [ ]:
RUN_ACTIVATION_CONTROLS = False
if RUN_ACTIVATION_CONTROLS:
    assert (V3_STUDY / 'locked-config.json').exists(), 'Restore or train v3 first.'
    RUN_NAME = MEMBER + '-activation-' + uuid.uuid4().hex[:8]
    RUN_DIR = ROOT / 'outputs' / RUN_NAME
    subprocess.run([sys.executable, 'code/activation_extension.py', '--study', str(V3_STUDY), '--plan', 'configs/activation-strengthening-v1.json', '--output', str(RUN_DIR)], check=True)


## 导出已完成的实验

检查 RUN_DIR 指向要保存的 outputs 子目录。EXPORT_RESULTS=True 后会打包结果，并写入 MEMBER、SEED、源码快照与逐文件 SHA。默认导出到临时 runtime，DOWNLOAD_ZIP=True 会触发下载。

离开临时 VM 前保存结果；也可将 EXPORT_DIRECTORY 指向自己已经授权的 Drive 文件夹。这里不会自动上传到其他成员的账号。若同时运行过训练和评价，请分别选择对应目录导出。


In [ ]:
EXPORT_RESULTS = False
EXPORT_DIRECTORY = ROOT / 'outputs' / 'exports'
DOWNLOAD_ZIP = True
if EXPORT_RESULTS:
    from colab_bundle import export_run
    is_v3 = bool(INDEPENDENT_CHECKPOINT_ARCHIVE or RUN_FRESH_V3 or RUN_ACTIVATION_CONTROLS)
    metadata = {'member': MEMBER, 'seed': None if is_v3 else SEED, 'training_seeds': [42, 43, 44] if is_v3 else [SEED], 'source_snapshot': SOURCE_SNAPSHOT, 'scope': 'Inspect exported manifests for the exact protocol and configuration.'}
    export = export_run(ROOT, RUN_DIR, EXPORT_DIRECTORY / (RUN_NAME + '.zip'), metadata)
    print('Export SHA-256:', export['sha256'])
    if DOWNLOAD_ZIP:
        from google.colab import files
        files.download(export['path'])
